# Fault detection on the Tennessee Eastman Process

**The idea in plain words:** learn what *normal* plant operation looks like, flag anything that does not look normal,
then ask which sensors caused the flag.

Run this notebook from the project folder (the one containing `tep_fd.py`).

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib
import tep_fd as T
import plots
try:
    from IPython.display import Image, display
except ImportError:                      # plain Python fallback
    def display(x): print(x)
    Image = lambda filename: filename

USE_SYNTHETIC = False        # True = fake data, only to check the code runs. NEVER quote its results.
DATA_DIR      = Path("data") # folder with TEP_FaultFree_Training / _Testing and TEP_Faulty_Testing (.csv or .RData)
TRAIN_RUNS    = 100          # fault-free runs to use (80% fit the model, 20% set thresholds)
TEST_RUNS     = 5            # runs per fault in the test set
FAULTS        = list(range(1, 21))
FAULT_START   = T.FAULT_START_TEST   # the fault begins after sample 160 (hour 8) in the test runs
OUT           = Path("results"); (OUT / "figures").mkdir(parents=True, exist_ok=True)

if USE_SYNTHETIC:
    from make_synthetic import make_synthetic
    DATA_DIR = make_synthetic(OUT / "synthetic_data")
    FAULTS, TRAIN_RUNS, TEST_RUNS = [1, 2, 3, 4, 5, 6], 60, 6

## Step 1 - Load normal data and split it
Some normal runs **fit** the model. A few other normal runs are held back to set the alarm thresholds,
so the thresholds are not tuned on the data the model has already seen.

In [ ]:
train = T.load_tep(T.find_tep_file(DATA_DIR, faulty=False, split="train"), faults=[0], max_run=TRAIN_RUNS)
runs = sorted(train["simulationRun"].unique())
n_fit = int(0.8 * len(runs))
X_fit = train[train["simulationRun"].isin(runs[:n_fit])][T.VARS].to_numpy(float)
X_val = train[train["simulationRun"].isin(runs[n_fit:])][T.VARS].to_numpy(float)
print(f"{len(runs)} normal runs -> fit on {len(X_fit):,} samples, thresholds from {len(X_val):,} samples")

## Step 2 - PCA model of normal operation
The plant has 52 sensors/valves that move together. PCA compresses them into a few main directions (`k` of them).
We keep enough directions to explain 90% of the variance.

In [ ]:
pca = T.PCAMonitor(variance=0.90, alpha=0.99).fit(X_fit, X_val)
print(f"k = {pca.k} components keep {pca.cum_var[pca.k-1]*100:.1f}% of the variance")
print(f"limits ({pca.limit_method}): T2 = {pca.t2_limit:.1f}, SPE = {pca.spe_limit:.1f}")
print(f"theoretical limits for comparison: T2 = {pca.t2_limit_theory:.1f}, SPE = {pca.spe_limit_theory:.1f}")
plots.plot_scree(pca, OUT / "figures/01_pca_variance.png")
display(Image(filename=str(OUT / "figures/01_pca_variance.png")))

## Step 3 - The two scores
- **T2** - the reading is *extreme* but still follows the normal pattern (e.g. everything runs hotter than usual).
- **SPE** - the reading *breaks* the normal pattern (e.g. two sensors that always move together suddenly do not).

A sample is an alarm if either score is above its control limit. We load the test runs and score them.

In [ ]:
test0 = T.load_tep(T.find_tep_file(DATA_DIR, faulty=False, split="test"), faults=[0], max_run=TEST_RUNS)
testf = T.load_tep(T.find_tep_file(DATA_DIR, faulty=True,  split="test"), faults=FAULTS, max_run=TEST_RUNS)
test = pd.concat([test0, testf], ignore_index=True)
X = test[T.VARS].to_numpy(float)

for k, v in pca.alarms(X).items():  test[k] = v
iforest = T.IForestMonitor(alpha=0.99).fit(X_fit, X_val)       # Step 4 baseline, fitted here so Step 5 can use it
for k, v in iforest.alarms(X).items(): test[k] = v

fault_to_plot = int(sorted(testf["faultNumber"].unique())[0])
g = test[(test["faultNumber"] == fault_to_plot) & (test["simulationRun"] == 1)].sort_values("sample")
plots.plot_monitoring(g, pca, fault_to_plot, 1, FAULT_START, OUT / "figures/03_monitoring_example.png")
display(Image(filename=str(OUT / "figures/03_monitoring_example.png")))

## Step 4 - Isolation Forest baseline
A standard ML anomaly detector: unusual points are separated from the rest in fewer random splits.
Its threshold is set the same way as for PCA (99th percentile of held-out normal scores), so the comparison is fair.
(It was fitted in the previous cell.)

## Step 5 - How well does it work?
- **FDR** - share of samples after the fault starts that raise an alarm (higher is better)
- **FAR** - share of samples before the fault starts that raise an alarm (lower is better)
- **Delay** - minutes from fault start to 3 consecutive alarms

In [ ]:
metrics = T.evaluate(test, fault_start=FAULT_START, consec=3)
metrics.to_csv(OUT / "metrics.csv", index=False)
display(metrics[metrics["method"].isin(["PCA (T2 or SPE)", "Isolation Forest"])]
        .pivot(index="fault", columns="method", values=["FDR_pct", "FAR_pct", "median_delay_min"]))
plots.plot_fdr(metrics, OUT / "figures/02_detection_rate_by_fault.png")
display(Image(filename=str(OUT / "figures/02_detection_rate_by_fault.png")))

## Step 6 - Root cause: which variables caused the alarm?
After an alarm, each score is split into per-variable contributions that add up exactly to the score.
The variables with the biggest share are the first suspects.

In [ ]:
post = test[(test["faultNumber"] == fault_to_plot) & (test["simulationRun"] == 1) & (test["sample"] > FAULT_START)]
which = "spe" if post["alarm_spe"].mean() >= post["alarm_t2"].mean() else "t2"
top = T.top_contributors(pca, post[T.VARS].to_numpy(float), which=which, top=10)
display(top)
plots.plot_contrib(top, fault_to_plot, which, OUT / "figures/04_contributions_example.png")
display(Image(filename=str(OUT / "figures/04_contributions_example.png")))

## Talking points
- Why PCA and not one threshold per sensor? Sensors are correlated, so a *combination* can be abnormal while each sensor looks fine.
- Why two scores? T2 catches unusual magnitude inside the normal pattern, SPE catches a broken pattern.
- Why held-out thresholds? The samples are autocorrelated, so textbook limits give the wrong false-alarm rate.
- Limitation: some TEP faults barely change the measurements and are hard for every method; TEP is continuous, batch processes need alignment first.